# Gridworld y su solución como MDPs

En este trabajo definiremos el ambiente de Gridworld y su solución como un MDP.
Gridworld es un ambiente clásico de prueba dentro del aprendizaje por refuerzo. Durante este taller definiremos el modelo básico del ambiente, que extenderemos incrementalmente de acuerdo a las necesidades del algoritmo de solución.

## Ambiente 🌎

El ambiente de gridworld se define como una cuadricula de `nxm`. El ambiente tiene obstaculos, es decir casillas por las cuales no puede pasar el agente. Al chocar con un obstaculo, el agente terminaría en el mismo estado inicial. Además, el ambiente tiene una casilla de inicio, y algunas casillas de salida. Un ejemplo del ambiente para el caso `3x4` se muestra a continuación.

![gridworld.png](./img/gridworld.png)

En este ejemplo del ambiente el agente comienza en la casilla inferior izquierda y tiene como objetivo llegar a la casilla de salida verde, con recompensa 1. La otra casilla de salida, tiene recompensa -1.

### Task 1.
#### ¿Cómo podemos codificar el ambiente?

De una definición completa del ambiente, como una clase de python llamada `Environment`, estableciendo:
1. Un atributo que define la cuadrícula (`board`). El ambiente recibirá una matriz como parámetro describiendo la cuadrícula en el momento de su creación. Definiremos las casillas por las que puede pasar el agente como casillas vacias, las casillas por las que no puede pasar el agente con un valor none `#` y las casillas de salida con el valor asociado a la recompensa definidas para cada una de ellas.
2. Un atributo `nrows` para almacenar la cantidad de filas de la cuadrícula.
3. Un atributo `ncols` para almacenar la cantidad de columnas de la cuadrícula.
4. Un atributo `initial_state` para almacenar el estado inicial del agente dentro del ambiente.
5. Un atributo con el estado actual (`current_state`) en el que se encuentra el agente. El valor de `current_state` se definirá como una tupla 
6. Un atributo `P` que guarda la matriz de probabilidades de cada una de las acciones para cada estado. Dicha matriz esta definida por parámetro.

Un ejemplo de la definición del tablero para el caso de 5x5 de la figura anterior se da a continuación.
```
board = [['', ' ', ' ',  '+1'],
         [' ', '#', ' ',  '-1'],
         ['S', ' ', ' ', ' ']]
```
En el ejemplo `S` denota el estado inicial y `'#'` la casilla prohibida (manejaremos esta convención para todos los ambientes de gridworld).

De forma similar a la definición del `board` la matriz de probabilidades `P` se define como:
```
P = [[[0.1, 0.1, 0, 0.8], [0.1, 0.1, 0, 0.8], [0.1, 0.1, 0, 0.8],  [1]],
         [[0.8, 0, 0.1, 0.1], '#', [0.8, 0, 0.1, 0.1],  [-1]],
         [[0.8, 0, 0.1, 0.1], [0.1, 0.1, 0.8, 0], [0.1, 0.1, 0.8, 0], [0.1, 0.1, 0.8, 0]]]
```
Para la definición de `P` vamos a entender cada una de las posiciones de la probabilidad en el orden (`'up', 'down', 'left', 'right'`). Adicionalmente, vamos a suponer que la casilla da la probabilidad de tal forma que el agente siempre toma la acción en dirección al objetivo (la acción que tiene probabilidad `0.8`). Por ejemplo, para la casilla superior izquierda la probabilidad de tomar la acción `right` y llegar a la casilla de arriba es de `0.1`, a la casilla de abajo con probabilidad `0.1` y a la casilla de la derecha con  probabilidad `0.8` (el agente nunca puede llegar a la casilla de la izquierda). En las casillas de salida, el agente solo tiene una posibilidad que es tomar la acción `exit` que le da la recompensa asociada a la casilla al agente.


#### Comportamiento del ambiente

Una vez definido el ambiente definimos su comportamiento. Para ello requerimos los siguientes métodos:
1. `get_current_state` que no recibe parámetros y retorna el estado actual (la casilla donde se encuentra el agente)
2. `get_posible_actions` que recibe el estado actual del agente como parámetro y retorna las acciones disponibles para dicho estado. Las acciones estarán dadas por su nombre (`'up', 'down', 'left', 'right'`) para las casillas normales y (`'exit'`) para las casillas de salida. Como convención definiremos que el agente siempre puede moverse en todas las direcciones, donde un movimiento en dirección de un obstáculo o los límites del ambiente no tienen ningún efecto visible en la posición del agente.
3. `do_action` que recibe como parámetro la acción a ejecutar y retorna el valor de la recompensa y el nuevo estado del agente, como un pareja `reward, new_state`. Note que `do_action` esta restringida por la matriz de probabilidad `P` para la ejecución real de las acciones.
4. `reset` que no recibe parámetros y restablece el ambiente a su estado inicial.
5. `is_terminal` que no recibe parámetros y determina si el agente está en el estado final o no. En nuestro caso, el estado final estará determinado por las casillas de salida (i.e., con un valor definido).

Teniendo en cuenta la definición del agente, genere un ambiente de `10x10` como se muestra a continuación.

![evaluacion.png](./img/evaluacion.png)

In [1]:
class Environment:
    EMPTY_CELL = ' '
    PROHIBITED_CELL = '#'
    INITIAL_CELL = 'S'

    def __init__(self, board, P):
        self.board = board
        self.nrows = len(board)
        self.ncols = len(board[0])
        self.initial_state = self.get_initial_state()
        self.current_state = self.initial_state
        self.P = P

    def get_initial_state(self):
        for i in range(self.nrows):
            for j in range(self.ncols):
                if self.board[i][j] == self.INITIAL_CELL:
                    return (i, j)
        return None
    
    def get_current_state(self):
        return self.current_state
    
    def get_posible_actions(self, state):
        i, j = state
        if self.board[i][j] == self.INITIAL_CELL or self.board[i][j] == self.EMPTY_CELL:
            return ['up', 'down', 'left', 'right']
        
        if self.board[i][j] != self.PROHIBITED_CELL:
            return ['exit']
        
        return []
        
    def do_action(self, action):
        i, j = self.current_state
        if action not in self.get_posible_actions(self.current_state):
            return 0, self.current_state
        
        # Obtener la probabilidad de la acción
        action_probabilities = self.P[i][j]

        if action_probabilities == self.PROHIBITED_CELL:
            print("You should not be here!")
            return 0, self.current_state
        
        if action == 'exit':
            reward = action_probabilities[0]  # Suponemos que la recompensa está en la primera posición de la lista
            self.current_state = self.initial_state
            return reward, self.current_state
        
        action_index = ['up', 'down', 'left', 'right'].index(action)
        action_probability = action_probabilities[action_index]

        if action_probability == 0:
            return 0, self.current_state
        
        # Determinar el nuevo estado basado en la acción y su probabilidad
        new_state = self.current_state
        if action == 'up':
            new_state = (max(i - 1, 0), j)
        elif action == 'down':
            new_state = (min(i + 1, self.nrows - 1), j)
        elif action == 'left':
            new_state = (i, max(j - 1, 0))
        elif action == 'right':
            new_state = (i, min(j + 1, self.ncols - 1))
        
        # Verificar si el nuevo estado es un obstáculo o no
        if self.board[new_state[0]][new_state[1]] == self.PROHIBITED_CELL:
            new_state = self.current_state
        
        # Actualizar el estado actual del agente
        self.current_state = new_state
        
        return 0, new_state
    
    def reset(self):
        self.current_state = self.initial_state

    def is_terminal(self):
        i, j = self.current_state
        return self.board[i][j] != self.EMPTY_CELL and self.board[i][j] != self.INITIAL_CELL


### Task 2.
Plantee el problema de MDP para cada una de las casillas. Especifique el estado de inicio, las transiciones y su probabilidad (suponiendo que todas las acciones sucede con probabilidad de 0.25) y los estados de fin con su recompensa.
¿Cómo serían las recompensas esperadas para cada estado?

In [ ]:
# Generamos el ambiente

board = [
    ['S', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
    [' ', '#', '#', '#', '#', ' ', '#', '#', '#', ' '],
    [' ', ' ', ' ', ' ', '#', ' ', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', '#', '-1', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', '#', '1', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', '#', ' ', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', '#', '-1', '-1', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
    [' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '],
 ]


p = []
for i in range(len(board)):
    row = []
    for j in range(len(board[i])):
        cell = board[i][j]
        if cell == Environment.PROHIBITED_CELL:
            row.append('#')
        elif cell == Environment.INITIAL_CELL:
            row.append([0.1, 0.1, 0.1, 0.8])  # El agente siempre toma la acción hacia el objetivo con probabilidad de 0.8
        elif cell == "1":
            row.append([1])  # No se pueden tomar acciones en celdas de recompensa
        elif cell == "-1":
            row.append([-1])  # No se pueden tomar acciones en celdas de penalización
        else:
            row.append([0.25, 0.25, 0.25, 0.25])  # En celdas vacías todas las acciones tienen la misma probabilidad
    p.append(row)



env = Environment(board, p)

#### Planteamiento del MDP para el gridworld 10x10
El problema de gridworld se puede modelar como un proceso de decisión de Markov (MDP) donde:
- **Estados**: Cada casilla del grid es un estado, representado por su posición (i, j).
- **Estado inicial**: La casilla marcada como 'S'.
- **Acciones**: ['up', 'down', 'left', 'right', 'exit'].
- **Transiciones**: Desde cada estado, el agente puede intentar moverse en cualquiera de las cuatro direcciones. Cada acción tiene probabilidad 0.25 de ejecutarse. Si la acción lleva a un obstáculo o fuera del grid, el agente permanece en el mismo estado.
- **Estados terminales**: Casillas con recompensa ("1" o "-1").
- **Recompensas**: Al llegar a una casilla terminal, el agente recibe la recompensa asociada. En los demás casos, la recompensa es 0.

#### Recompensas esperadas para cada estado
La recompensa esperada para cada estado se puede calcular considerando la probabilidad de llegar a un estado terminal desde cada posición, bajo la política uniforme (todas las acciones con probabilidad 0.25).
Para cada estado no terminal, la recompensa esperada es la suma de las recompensas de los estados alcanzables ponderadas por la probabilidad de transición.

A continuación se muestra un ejemplo de cómo calcular las recompensas esperadas para cada estado usando el grid definido en la Task 1.

In [5]:
import pandas as pd

# Algoritmo para generar el DataFrame de definición de MDP
def generar_mdp_dataframe(env):
    estados = []
    acciones = []
    estados_f = []
    probabilidades = []
    recompensas = []
    for i in range(env.nrows):
        for j in range(env.ncols):
            estado = (i, j)
            cell = env.board[i][j]
            posibles_acciones = env.get_posible_actions(estado)
            P_cell = env.P[i][j]
            for idx, accion in enumerate(posibles_acciones):
                if accion == 'exit':
                    # Solo para celdas terminales
                    estados.append(estado)
                    acciones.append('exit')
                    estados_f.append('terminal')
                    probabilidades.append(1.0)
                    # Recompensa de la celda
                    recompensa = float(cell)
                    recompensas.append(recompensa)
                else:
                    # Acciones de movimiento
                    if cell == env.PROHIBITED_CELL or cell == '1' or cell == '-1':
                        continue
                    # Probabilidad de la acción
                    prob = P_cell[idx]
                    if prob == 0:
                        continue
                    # Calcular el estado final
                    ni, nj = i, j
                    if accion == 'up':
                        ni = max(i - 1, 0)
                    elif accion == 'down':
                        ni = min(i + 1, env.nrows - 1)
                    elif accion == 'left':
                        nj = max(j - 1, 0)
                    elif accion == 'right':
                        nj = min(j + 1, env.ncols - 1)
                    # Si es obstáculo, se queda en el mismo estado
                    if env.board[ni][nj] == env.PROHIBITED_CELL:
                        ni, nj = i, j
                    estados.append(estado)
                    acciones.append(accion)
                    estados_f.append((ni, nj))
                    probabilidades.append(prob)
                    # Recompensa de la celda destino
                    cell_dest = env.board[ni][nj]
                    if cell_dest == '1' or cell_dest == '-1':
                        recompensa = float(cell_dest)
                    else:
                        recompensa = 0.0
                    recompensas.append(recompensa)
    df = pd.DataFrame({
        'Estado inicial (s)': estados,
        'Acción (a)': acciones,
        "Estado final (s')": estados_f,
        "Probabilidad (P(s'|s,a))": probabilidades,
        "Recompensa (R(s'))": recompensas
    })
    return df

# Ejemplo de uso:
mdp_df = generar_mdp_dataframe(env)
mdp_df.to_csv("mdp_definition.csv", index=False)
mdp_df.head(30)

,Estado inicial (s),Acción (a),Estado final (s'),"Probabilidad (P(s'|s,a))",Recompensa (R(s'))
0,"(0, 0)",up,"(0, 0)",0.10,0.0
1,"(0, 0)",down,"(1, 0)",0.10,0.0
2,"(0, 0)",left,"(0, 0)",0.10,0.0
3,"(0, 0)",right,"(0, 1)",0.80,0.0
4,"(0, 1)",up,"(0, 1)",0.25,0.0
5,"(0, 1)",down,"(1, 1)",0.25,0.0
6,"(0, 1)",left,"(0, 0)",0.25,0.0
7,"(0, 1)",right,"(0, 2)",0.25,0.0
8,"(0, 2)",up,"(0, 2)",0.25,0.0
9,"(0, 2)",down,"(1, 2)",0.25,0.0


### Task 3.
Bajo la definción del problema anterior, suponga que cada acción tiene una probabilidad de éxito de 60%, con probabilidad de 20% se ejecutará la sigiente acción (en dirección de las manesillas del reloj), con probabilidad de 10% se ejecutará la sigiente acción (en contra de las manesillas del reloj) y con probabilidad de 10% no pasará nada. Bajo estas condiciones, ¿Cómo serían las recompensas esperadas para cada estado? 

Codifique el ambiente para el gridworld de `10x10` utilizando esta función de probabilidad. En esta codificación del ambiente no es necesario pasar la matriz `P` como parámetro, pero esta información se debe tener en cuenta en la función `do_action`.

Tenga en cuenta que la calidad del programa que entreguen será tenida en cuentra dentro de la calificación.

In [7]:
import random

class Clock_Environment(Environment):
    def __init__(self, board):
        super().__init__(board, P=None)  # No requiere matriz P
        self.P = None  # No se usa

    def do_action(self, action):
        i, j = self.current_state
        acciones = ['up', 'right', 'down', 'left']
        if action not in self.get_posible_actions(self.current_state):
            return 0, self.current_state

        probas = [0.6, 0.2, 0.1, 0.1]  # éxito, siguiente (horario), anterior (antihorario), nada
        idx = acciones.index(action)
        r = random.random()
        if r < probas[0]:
            accion_real = action
        elif r < probas[0] + probas[1]:
            accion_real = acciones[(idx + 1) % 4]
        elif r < probas[0] + probas[1] + probas[2]:
            accion_real = acciones[(idx - 1) % 4]
        else:
            accion_real = None  # No pasa nada

        if accion_real is None:
            return 0, self.current_state

        # Calcular nuevo estado
        ni, nj = i, j
        if accion_real == 'up':
            ni = max(i - 1, 0)
        elif accion_real == 'down':
            ni = min(i + 1, self.nrows - 1)
        elif accion_real == 'left':
            nj = max(j - 1, 0)
        elif accion_real == 'right':
            nj = min(j + 1, self.ncols - 1)

        # Si es obstáculo, no se mueve
        if self.board[ni][nj] == self.PROHIBITED_CELL:
            ni, nj = i, j

        # Si es terminal, recompensa y reset
        if self.board[ni][nj] == '1' or self.board[ni][nj] == '-1':
            reward = float(self.board[ni][nj])
            self.current_state = self.initial_state
            return reward, self.current_state

        self.current_state = (ni, nj)
        return 0, self.current_state


In [8]:
import pandas as pd

def generar_mdp_dataframe_clock(env):
    estados = []
    acciones = []
    estados_f = []
    probabilidades = []
    recompensas = []
    acciones_lista = ['up', 'right', 'down', 'left']
    transiciones = {
        'up':    [(0.6, 'up'), (0.2, 'right'), (0.1, 'left'), (0.1, None)],
        'right': [(0.6, 'right'), (0.2, 'down'), (0.1, 'up'), (0.1, None)],
        'down':  [(0.6, 'down'), (0.2, 'left'), (0.1, 'right'), (0.1, None)],
        'left':  [(0.6, 'left'), (0.2, 'up'), (0.1, 'down'), (0.1, None)]
    }
    for i in range(env.nrows):
        for j in range(env.ncols):
            estado = (i, j)
            posibles_acciones = env.get_posible_actions(estado)
            for accion in acciones_lista:
                if accion not in posibles_acciones:
                    continue
                for prob, accion_real in transiciones[accion]:
                    if accion_real is None:
                        ni, nj = i, j
                    else:
                        ni, nj = i, j
                        if accion_real == 'up':
                            ni = max(i - 1, 0)
                        elif accion_real == 'down':
                            ni = min(i + 1, env.nrows - 1)
                        elif accion_real == 'left':
                            nj = max(j - 1, 0)
                        elif accion_real == 'right':
                            nj = min(j + 1, env.ncols - 1)
                        # Si es obstáculo, no se mueve
                        if env.board[ni][nj] == env.PROHIBITED_CELL:
                            ni, nj = i, j
                    # Si es terminal, recompensa y reset
                    cell_dest = env.board[ni][nj]
                    if cell_dest == '1' or cell_dest == '-1':
                        recompensa = float(cell_dest)
                        estado_final = 'terminal'
                    else:
                        recompensa = 0.0
                        estado_final = (ni, nj)
                    estados.append(estado)
                    acciones.append(accion)
                    estados_f.append(estado_final)
                    probabilidades.append(prob)
                    recompensas.append(recompensa)
    df = pd.DataFrame({
        'Estado inicial (s)': estados,
        'Acción (a)': acciones,
        "Estado final (s')": estados_f,
        "Probabilidad (P(s'|s,a))": probabilidades,
        "Recompensa esperada (R(s'))": recompensas
    })
    return df

# Ejemplo de uso:
clock_env = Clock_Environment(board)
mdp_df_clock = generar_mdp_dataframe_clock(clock_env)
mdp_df_clock.head(30)

,Estado inicial (s),Acción (a),Estado final (s'),"Probabilidad (P(s'|s,a))",Recompensa esperada (R(s'))
0,"(0, 0)",up,"(0, 0)",0.6,0.0
1,"(0, 0)",up,"(0, 1)",0.2,0.0
2,"(0, 0)",up,"(0, 0)",0.1,0.0
3,"(0, 0)",up,"(0, 0)",0.1,0.0
4,"(0, 0)",right,"(0, 1)",0.6,0.0
5,"(0, 0)",right,"(1, 0)",0.2,0.0
6,"(0, 0)",right,"(0, 0)",0.1,0.0
7,"(0, 0)",right,"(0, 0)",0.1,0.0
8,"(0, 0)",down,"(1, 0)",0.6,0.0
9,"(0, 0)",down,"(0, 0)",0.2,0.0


### Task 4. 
Defina una situación de la vide real (de su escogencia) como un MDP.

### Gestión de inventario en una tienda

Una tienda debe decidir cada semana cuántas unidades de un producto ordenar, considerando que la demanda es incierta.

#### Estados (S)

El estado sería la cantidad actual de inventario (0, 1, 2, ..., N)

Puede incluir también si hay pedidos pendientes

Por ejemplo:

Estado = número de unidades en stock al inicio de la semana.

#### Acciones (A)

* Ordenar 0 unidades
* Ordenar 1 unidad
* Ordenar 2 unidades
* ...
* Ordenar hasta un máximo permitido

Estas decisiones determinan cuántas unidades se agregarán al inventario al inicio de la semana.

#### Probabilidades de transición (P)

El inventario de la siguiente semana depende de:

* Inventario actual
* Cantidad ordenada
* Demanda aleatoria

Por ejemplo:

Si tengo 5 unidades y ordeno 3 → tengo 8.
Si la demanda aleatoria es 6 → el siguiente estado será 2.
Si la demanda es mayor que 8 → el inventario queda en 0.

La incertidumbre viene de la distribución probabilística de la demanda.

#### Recompensas (R)

Puede definirse como la ganancia neta semanal: Ingresos - Costos

*  (+) Ganancia por cada unidad vendida
*  (-) Costo de ordenar productos
*  (-) Costo de almacenamiento
*  (-) Penalización por falta de stock